In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    avg,
    desc,
    asc,
    regexp_replace,
    round,
    when,
    to_date,
    min as spark_min,
    max as spark_max
)

spark = SparkSession.builder \
    .appName("Lab4 Airbnb Team16 Spark DataFrame Analysis") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.range(10).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 07:12:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 07:12:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/06 07:12:44 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:12:59 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:13:14 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:13:29 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your clu

Py4JError: An error occurred while calling o33.showString

26/05/06 07:15:14 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:15:29 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:15:44 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:15:59 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:16:14 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/06 07:16:29 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

In [ ]:
BASE_PATH = "/home/ubuntu/airbnb-spark-lab/data/raw"
OUTPUT_TABLE_PATH = "/home/ubuntu/airbnb-spark-lab/outputs/tables"
OUTPUT_CHART_PATH = "/home/ubuntu/airbnb-spark-lab/outputs/charts"

print("Base path:", BASE_PATH)
print("Output table path:", OUTPUT_TABLE_PATH)
print("Output chart path:", OUTPUT_CHART_PATH)

In [ ]:
listings = spark.read.csv(
    f"{BASE_PATH}/listings.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

calendar = spark.read.csv(
    f"{BASE_PATH}/calendar.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

reviews = spark.read.csv(
    f"{BASE_PATH}/reviews.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

neighbourhoods = spark.read.csv(
    f"{BASE_PATH}/neighbourhoods.csv",
    header=True,
    inferSchema=True
)

print("Listings:", listings.count())
print("Calendar:", calendar.count())
print("Reviews:", reviews.count())
print("Neighbourhoods:", neighbourhoods.count())

In [ ]:
listings.show(5, truncate=False)

In [ ]:
calendar.show(5, truncate=False)

In [ ]:
reviews.show(5, truncate=False)

In [ ]:
neighbourhoods.show(5, truncate=False)

In [ ]:
neighbourhoods.show(5, truncate=False)

The listings dataset contains detailed information about Airbnb listings, including listing identifiers, host information, location, room type, price, availability, review metrics, and other property-related attributes. The schema inspection step helps identify important fields for subsequent cleaning and analysis.

In [ ]:
listings_clean = listings.withColumn(
    "price_clean",
    regexp_replace(col("price"), "[$,]", "").cast("double")
)

important_cols = [
    "id",
    "name",
    "host_id",
    "host_name",
    "neighbourhood_group_cleansed",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "price",
    "price_clean",
    "minimum_nights",
    "maximum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "calculated_host_listings_count"
]

existing_cols = [c for c in important_cols if c in listings_clean.columns]
listings_clean = listings_clean.select(*existing_cols)

listings_clean.show(5, truncate=False)

In [ ]:
null_report = []

for c in listings_clean.columns:
    null_count = listings_clean.where(col(c).isNull()).count()
    null_report.append((c, null_count))

null_df = spark.createDataFrame(null_report, ["column_name", "null_count"])
null_df.orderBy(desc("null_count")).show(30)

In [ ]:
if "neighbourhood_group_cleansed" in listings_clean.columns:
    region_col = "neighbourhood_group_cleansed"
else:
    region_col = "neighbourhood_cleansed"

print("Using region column:", region_col)

In [ ]:
host_listing_count = listings_clean.groupBy("host_id", "host_name") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count"))

host_listing_count.show(20, truncate=False)

if "calculated_host_listings_count" in listings_clean.columns:
    listings_clean.select(
        spark_max("calculated_host_listings_count").alias("max_calculated_host_listings_count")
    ).show()

In [ ]:
region_count = listings_clean.groupBy(region_col) \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count"))

region_count.show()

In [ ]:
import matplotlib.pyplot as plt

region_pd = region_count.toPandas()

plt.figure(figsize=(8, 8))
plt.pie(
    region_pd["listing_count"],
    labels=region_pd[region_col],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Listings by Region")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/listings_by_region.png")
plt.show()